# 02 — Orbite proche d'un trou noir (Schwarzschild)

Dans le notebook 01, la précession GR de Mercure était de **5×10⁻⁷ rad/orbite** — minuscule, invisible à l'œil. Ici on passe au **régime relativiste fort** : une étoile en orbite autour d'un trou noir stellaire. La précession devient de l'ordre de **plusieurs dizaines de degrés par orbite**, et de nouveaux phénomènes apparaissent que Newton ne prédit pas du tout :

## Phénomènes spécifiques à la GR

1. **Précession énorme** du périhélie, plusieurs degrés à dizaines de degrés par orbite.
2. **ISCO** (*Innermost Stable Circular Orbit*) : à $r = 6GM/c^2 = 3r_s$, il n'existe plus d'orbite circulaire stable. **En Newton, n'importe quel rayon supporte une orbite circulaire — pas en GR.**
3. **Plongée vers l'horizon** : sous certaines conditions, l'orbite spirale jusqu'à $r = r_s$ et le corps tombe — comportement absent en mécanique newtonienne.

## Constantes du trou noir

On choisit un trou noir stellaire de masse $M = 10\,M_\odot$ :
- $r_s = 2GM/c^2 \approx 29.5$ km
- ISCO = $3 r_s \approx 88.6$ km

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

from src.config import G, C, M_SUN
from src.newtonian import simulate_two_body
from src.relativity import (
    simulate_orbit_schwarzschild,
    schwarzschild_radius,
    isco_radius,
)

plt.rcParams['figure.dpi'] = 110

# Paramètres du trou noir
M_BH = 10 * M_SUN
rs   = schwarzschild_radius(M_BH)
risco = isco_radius(M_BH)

print(f"Trou noir : {M_BH/M_SUN:.0f} M_sun")
print(f"  r_Schwarzschild = {rs/1000:.2f} km")
print(f"  r_ISCO          = {risco/1000:.2f} km  ({risco/rs:.0f} rs)")

## 1. Orbite stable mais relativiste (r₍peri₎ = 10 r_s)

On part avec un *périhélie* à 10 fois le rayon de Schwarzschild, avec des conditions initiales képlériennes (e=0.5). Newton donne une belle ellipse fermée. La GR donne… autre chose.

In [ ]:
def setup_orbit(r_peri_in_rs, e):
    """Retourne (r_peri, v_peri, T_kepler) pour une orbite de demi-grand axe a = r_peri/(1-e)."""
    r_peri = r_peri_in_rs * rs
    a_orb = r_peri / (1 - e)
    v_peri = np.sqrt(G * M_BH * (1 + e) / (a_orb * (1 - e)))
    T = 2 * np.pi * np.sqrt(a_orb**3 / (G * M_BH))
    return r_peri, v_peri, T

r_peri, v_peri, T_kepler = setup_orbit(r_peri_in_rs=10, e=0.5)
n_orbits = 3

res_n  = simulate_two_body(M_BH, r_peri, v_peri, t_max=n_orbits*T_kepler, n_steps=5000)
res_gr = simulate_orbit_schwarzschild(M_BH, r_peri, v_peri, n_orbits=n_orbits, n_points=5000)

# Précession GR
if len(res_gr.phi_perihelia) >= 2:
    delta_phi = np.diff(res_gr.phi_perihelia)[0] - 2*np.pi
    print(f"Précession GR par orbite : {np.degrees(delta_phi):.2f}°")
print(f"v_peri / c = {v_peri/C:.3f}  (régime relativiste !)\n")

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(res_n.x/rs, res_n.y/rs, label='Newton', color='#1f77b4', lw=1.4)
ax.plot(res_gr.x/rs, res_gr.y/rs, label='Schwarzschild (GR)', color='#d62728', lw=1.4)

# Horizon (rs) et ISCO
ax.add_patch(plt.Circle((0,0), 1, color='black', zorder=5, label='Horizon (rs)'))
ax.add_patch(plt.Circle((0,0), 3, color='gray', fill=False, ls='--', lw=1, label='ISCO (3rs)'))

ax.set_aspect('equal'); ax.set_xlabel('x [rs]'); ax.set_ylabel('y [rs]')
ax.set_title(f'Étoile autour d\'un trou noir 10 M⊙ — {n_orbits} orbites képlériennes')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Observation** : la trajectoire GR **rosace** — chaque retour au périhélie s'effectue avec un décalage angulaire visible. Sur 3 orbites Newton, l'ellipse est parfaitement fermée ; en GR, on voit clairement la précession (~50°/orbite ici).

Note : le r_max de l'orbite GR est *plus petit* que celui de l'orbite newtonienne, à conditions initiales identiques. C'est un effet d'énergie : la GR « lie » plus fortement le corps que Newton.

## 2. Tendre vers l'ISCO — l'orbite devient instable

À mesure qu'on se rapproche de l'ISCO ($3r_s$), les orbites « ferme-ouvertes » disparaissent. Voyons un cas extrême.

In [ ]:
r_peri, v_peri, T_kepler = setup_orbit(r_peri_in_rs=4, e=0.5)
n_orbits = 4

res_n  = simulate_two_body(M_BH, r_peri, v_peri, t_max=n_orbits*T_kepler, n_steps=10000)
res_gr = simulate_orbit_schwarzschild(M_BH, r_peri, v_peri, n_orbits=n_orbits, n_points=10000)

print(f"v_peri / c = {v_peri/C:.3f}")
print(f"Newton : r reste entre {res_n.r.min()/rs:.2f} rs et {res_n.r.max()/rs:.2f} rs (orbite stable)")
print(f"GR     : r descend jusqu'à {res_gr.r.min()/rs:.2f} rs  (proche de l'horizon !)\n")

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(res_n.x/rs, res_n.y/rs, label='Newton', color='#1f77b4', lw=1.0)
ax.plot(res_gr.x/rs, res_gr.y/rs, label='Schwarzschild (GR)', color='#d62728', lw=1.0)
ax.add_patch(plt.Circle((0,0), 1, color='black', zorder=5))
ax.add_patch(plt.Circle((0,0), 3, color='gray', fill=False, ls='--', lw=1))
ax.set_aspect('equal'); ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
ax.set_xlabel('x [rs]'); ax.set_ylabel('y [rs]')
ax.set_title('Près de l\'ISCO : Newton reste régulier, GR plonge vers l\'horizon')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

**En Newton**, même très proche du centre, l'orbite reste une ellipse fermée — la gravité newtonienne n'a pas de notion d'horizon ni d'orbite instable.

**En GR**, le potentiel effectif a une « barrière centrifuge » qui s'effondre près de l'ISCO. Avec des conditions initiales képlériennes (mais à 4 rs de l'horizon), le corps n'a pas assez de moment angulaire pour rester en orbite : il **plonge** vers l'horizon en spirale.

## 3. Explorateur interactif

Joue avec le rayon initial pour voir la transition orbite stable ↔ plongée.

In [ ]:
def explore(r_peri_in_rs=8.0, e=0.4, n_orbits=4):
    r_peri, v_peri, T_kepler = setup_orbit(r_peri_in_rs, e)

    res_n  = simulate_two_body(M_BH, r_peri, v_peri, t_max=n_orbits*T_kepler, n_steps=8000)
    res_gr = simulate_orbit_schwarzschild(M_BH, r_peri, v_peri, n_orbits=n_orbits, n_points=8000)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(res_n.x/rs, res_n.y/rs, label='Newton', color='#1f77b4', lw=1.0)
    ax.plot(res_gr.x/rs, res_gr.y/rs, label='Schwarzschild', color='#d62728', lw=1.0)
    ax.add_patch(plt.Circle((0,0), 1, color='black', zorder=5))
    ax.add_patch(plt.Circle((0,0), 3, color='gray', fill=False, ls='--', lw=1))
    ax.set_aspect('equal')
    lim = max(8, 1.2 * res_n.r.max()/rs)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel('x [rs]'); ax.set_ylabel('y [rs]')
    title = f'r_peri = {r_peri_in_rs:.1f} rs, e = {e:.2f}, v/c = {v_peri/C:.2f}'
    if len(res_gr.phi_perihelia) >= 2:
        dphi = np.diff(res_gr.phi_perihelia)[0] - 2*np.pi
        title += f'  |  précession GR = {np.degrees(dphi):.1f}°/orb'
    ax.set_title(title)
    ax.legend(); ax.grid(alpha=0.3)
    plt.show()

interact(
    explore,
    r_peri_in_rs=FloatSlider(value=8.0, min=3.5, max=30.0, step=0.5, description='r_peri [rs]'),
    e=FloatSlider(value=0.4, min=0.0, max=0.85, step=0.05, description='excentricité'),
    n_orbits=IntSlider(value=4, min=1, max=10, step=1, description='# orbites'),
);

## 4. Précession en fonction du rayon

On balaye r_peri et on mesure la précession GR par orbite — pour quantifier à quelle distance la GR « s'allume » vraiment.

In [ ]:
r_peris = np.linspace(5, 50, 30)    # de 5 rs à 50 rs
precessions_deg = []

for r_p_in_rs in r_peris:
    r_peri, v_peri, _ = setup_orbit(r_p_in_rs, e=0.3)
    res = simulate_orbit_schwarzschild(M_BH, r_peri, v_peri, n_orbits=2, n_points=3000)
    if len(res.phi_perihelia) >= 2:
        dphi = np.diff(res.phi_perihelia)[0] - 2*np.pi
        precessions_deg.append(np.degrees(dphi))
    else:
        precessions_deg.append(np.nan)

# Théorie faible champ : Δφ ≈ 6π·GM/(c²·a·(1-e²)) en rad/orbite
a_theory = r_peris * rs / (1 - 0.3)
delta_theory = 6*np.pi * G*M_BH / (C**2 * a_theory * (1 - 0.3**2))
delta_theory_deg = np.degrees(delta_theory)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_peris, precessions_deg, 'o-', label='Simulation GR', color='#d62728')
ax.plot(r_peris, delta_theory_deg, '--', label=r'Théorie faible champ : $6\pi GM/(c^2 a(1-e^2))$', color='black')
ax.axvline(3, color='gray', ls=':', alpha=0.7, label='ISCO (3rs)')
ax.set_xlabel(r'$r_{\rm peri}$ [rs]')
ax.set_ylabel('Précession par orbite [°]')
ax.set_yscale('log')
ax.set_title('Précession GR : régime fort vs formule faible champ')
ax.legend(); ax.grid(alpha=0.3, which='both')
plt.show()

**Lecture du graphe** :
- Loin du trou noir ($r > 30 r_s$), la formule analytique faible champ est en parfait accord avec la simulation.
- À mesure qu'on se rapproche, la simulation **diverge vers le haut** : les corrections d'ordre supérieur en $GM/(c^2 r)$ deviennent dominantes. Près de l'ISCO, la précession explose.
- Newton, lui, donne **toujours 0** — peu importe la distance.

## Bilan

| Phénomène | Newton | Schwarzschild (GR) |
|---|---|---|
| Précession du périhélie | 0 | jusqu'à dizaines de degrés/orbite |
| ISCO | n'existe pas | à $3 r_s$ |
| Plongée vers l'horizon | impossible | possible dès que h est trop faible |
| Orbite circulaire à tout rayon | OK | uniquement si $r > 3 r_s$ |

La gravité newtonienne est universelle pour le quotidien, mais en champ fort (trou noir, étoile à neutrons) elle manque qualitativement plusieurs phénomènes essentiels.

**Suite** : notebook 03 — déviation de la lumière (géodésiques nulles, lentille gravitationnelle).